In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

root = Path.cwd()/'institutional-roi-analysis'
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

In [ ]:
def get_with_retries(url, params, tries=5, timeout=30):
    last = None
    for i in range(tries):
        r = requests.get(
            url,
            params=params,
            timeout=timeout,
            headers={"Accept": "application/json"},
        )
        if r.status_code < 500 and r.status_code != 429:
            return r
        if r.status_code == 429:
            time.sleep(5 * (i + 1))
            continue
        last = r
        time.sleep((2 ** i) + random.random())
    return last


def get_json_or_raise(response: requests.Response):
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        ct = response.headers.get("Content-Type", "")
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"HTTP {response.status_code} for {response.url}\n"
            f"Content-Type: {ct}\n"
            f"Body preview:\n{body_preview}"
        ) from e

    ct = response.headers.get("Content-Type", "")
    if "json" not in ct.lower():
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"Expected JSON but got Content-Type: {ct}\n"
            f"URL: {response.url}\n"
            f"Body preview:\n{body_preview}"
        )

    try:
        return response.json()
    except json.JSONDecodeError as e:
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"JSON decode failed for {response.url}\n"
            f"Body preview:\n{body_preview}"
        ) from e


def collect_school_level(state: str = "FL", per_page: int = 100, api_key: str = "") -> pd.DataFrame:
    fields = ",".join([
        "id",
        "school.name",
        "unit"
        "latest.academics.program_percentage.agriculture",
        "latest.academics.program_percentage.resources",
        "latest.academics.program_percentage.architecture",
        "latest.academics.program_percentage.ethnic_cultural_gender",
        "latest.academics.program_percentage.communication",
        "latest.academics.program_percentage.communications_technology",
        "latest.academics.program_percentage.computer",
        "latest.academics.program_percentage.personal_culinary",
        "latest.academics.program_percentage.education",
        "latest.academics.program_percentage.engineering",
        "latest.academics.program_percentage.engineering_technology",
        "latest.academics.program_percentage.language",
        "latest.academics.program_percentage.family_consumer_science",
        "latest.academics.program_percentage.legal",
        "latest.academics.program_percentage.english",
        "latest.academics.program_percentage.humanities",
        "latest.academics.program_percentage.library",
        "latest.academics.program_percentage.biological",
        "latest.academics.program_percentage.mathematics",
        "latest.academics.program_percentage.military",
        "latest.academics.program_percentage.multidiscipline",
        "latest.academics.program_percentage.parks_recreation_fitness",
        "latest.academics.program_percentage.philosophy_religious",
        "latest.academics.program_percentage.theology_religious_vocation",
        "latest.academics.program_percentage.physical_science",
        "latest.academics.program_percentage.science_technology",
        "latest.academics.program_percentage.psychology",
        "latest.academics.program_percentage.security_law_enforcement",
        "latest.academics.program_percentage.public_administration_social_service",
        "latest.academics.program_percentage.social_science",
        "latest.academics.program_percentage.construction",
        "latest.academics.program_percentage.mechanic_repair_technology",
        "latest.academics.program_percentage.precision_production",
        "latest.academics.program_percentage.transportation",
        "latest.academics.program_percentage.visual_performing",
        "latest.academics.program_percentage.health",
        "latest.academics.program_percentage.business_marketing",
        "latest.academics.program_percentage.history",
        "latest.school.instructional_expenditure_per_fte",
        "latest.school.faculty_salary",
        "latest.school.ft_faculty_rate",
        "latest.academics.program_reporter.programs_offered",
        "latest.student.demographics.student_faculty_ratio",
        "latest.school.endowment.begin",
        "latest.school.endowment.end",
        "latest.school.dolflag",
        "latest.admissions.admission_rate.overall",
        "latest.school.open_admissions_policy",
    ])

    params = {
        "api_key": SCORECARD_KEY,
        "school.state": state,
        "fields": fields,
        "per_page": str(per_page),
        "page": "0",
    }

    response = get_with_retries(BASE_URL, params=params, timeout=30)
    data = get_json_or_raise(response)

    total = int(data["metadata"]["total"])
    per_page_actual = int(data["metadata"]["per_page"])
    total_pages = math.ceil(total / per_page_actual)

    rows = []

    for page in range(total_pages):
        params["page"] = str(page)
        response = get_with_retries(BASE_URL, params=params, timeout=30)
        data = get_json_or_raise(response)
        rows.extend(data.get("results", []))

    df = pd.json_normalize(rows)

    print(f"Total pages fetched: {total_pages}")
    print(f"Total schools ingested: {len(df)}")
    return df

In [4]:
tdf=collect_school_level()

Total pages fetched: 4
Total schools ingested: 374


In [5]:
save(tdf,file_name="inst_residual_driver")
tdf=clean(tdf)

Numeric columns: Index(['program_percentage_agriculture', 'program_percentage_resources',
       'program_percentage_architecture',
       'program_percentage_ethnic_cultural_gender',
       'program_percentage_communication',
       'program_percentage_communications_technology',
       'program_percentage_computer', 'program_percentage_personal_culinary',
       'program_percentage_education', 'program_percentage_engineering',
       'program_percentage_engineering_technology',
       'program_percentage_language',
       'program_percentage_family_consumer_science',
       'program_percentage_legal', 'program_percentage_english',
       'program_percentage_humanities', 'program_percentage_library',
       'program_percentage_biological', 'program_percentage_mathematics',
       'program_percentage_military', 'program_percentage_multidiscipline',
       'program_percentage_parks_recreation_fitness',
       'program_percentage_philosophy_religious',
       'program_percentage_theology

C:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis\src\ira\clean\clean_scorecard.py:8: FutureWarning:

The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.



In [6]:
tdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 50 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   program_percentage_agriculture                           306 non-null    float64
 1   program_percentage_resources                             306 non-null    float64
 2   program_percentage_architecture                          306 non-null    float64
 3   program_percentage_ethnic_cultural_gender                306 non-null    float64
 4   program_percentage_communication                         306 non-null    float64
 5   program_percentage_communications_technology             306 non-null    float64
 6   program_percentage_computer                              306 non-null    float64
 7   program_percentage_personal_culinary                     306 non-null    float64
 8   program_percentage_education  

In [428]:
driver_df=tdf.copy()

### Feature Engineering: Program Composition

We explored aggregating program percentage features into broader categories (e.g., STEM, Health, Humanities) to reduce dimensionality and simplify interpretation.

However, this approach was not adopted at this stage. Many of the program categories capture meaningful distinctions between institutions (e.g., engineering vs. computer science vs. biological sciences), and grouping them introduced a loss of potentially valuable signal.

Since the objective of this phase is to identify factors that explain variability in institutional performance, preserving this level of granularity was prioritized.

Feature grouping may be revisited in later iterations if dimensionality reduction or interpretability becomes a limiting factor.

In [429]:
driver_df["has_endowment"] = (
    driver_df["endowment_begin"].notna() &
    driver_df["endowment_end"].notna()
).astype(int)

In [430]:
driver_df["has_endowment"].describe()

count    374.000000
mean       0.200535
std        0.400937
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: has_endowment, dtype: float64

In [431]:
residual_df = pd.read_csv(root/"data"/"clean"/"scorecard"/"clean_inst_residual_FL.csv")
display(residual_df.info())
residual_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2052 entries, 0 to 2051
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   code                       2052 non-null   int64  
 1   unit_id                    2052 non-null   int64  
 2   distance                   2052 non-null   int64  
 3   school_type                2052 non-null   object 
 4   credential_level           2052 non-null   int64  
 5   school_name                2052 non-null   object 
 6   locale                     2052 non-null   int64  
 7   carnegie_size_setting      2052 non-null   int64  
 8   admission_rate_overall     1168 non-null   float64
 9   median_family_income       2044 non-null   float64
 10  students_with_pell_grant   1899 non-null   float64
 11  open_admissions_policy     2051 non-null   float64
 12  age_entry                  2044 non-null   float64
 13  title_iv_eligibility_type  2052 non-null   int64

None

,code,unit_id,distance,school_type,credential_level,school_name,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,actual,pred,title,4_yr_working_count,error,school_count,confidence,pct_error,score,inst_n,inst_effective_n,error_mean,inst_wr_mean,adj_inst_wr_mean
0,1205,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,22265.0,32497.172,Culinary Arts and Related Services.,28,-10232.171875,6,medium,-0.314863,-0.300975,206,36.68932,-843.318359,-2415.625,455.11542
1,4603,132374,2,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,41177.0,41086.582,Electrical and Power Transmission Installers.,27,90.417969,5,low,0.002201,0.002100,206,36.68932,-843.318359,-2415.625,455.11542
2,4702,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,47325.0,41659.600,"Heating, Air Conditioning, Ventilation and Ref...",48,5665.398438,13,medium,0.135993,0.132757,206,36.68932,-843.318359,-2415.625,455.11542
3,4706,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,42839.0,51169.758,Vehicle Maintenance and Repair Technologies/Te...,28,-8330.757813,24,high,-0.162806,-0.155625,206,36.68932,-843.318359,-2415.625,455.11542
4,5108,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,33081.0,34282.832,Allied Health and Medical Assisting Services.,29,-1201.832031,33,high,-0.035056,-0.033570,206,36.68932,-843.318359,-2415.625,455.11542


In [432]:
join_cols=["unit_id"]
for col in join_cols:
    driver_df[col] = driver_df[col].astype(str).str.strip()
    residual_df[col] = residual_df[col].astype(str).str.strip()


### Target Variable Selection

While a composite scoring metric was developed to rank program-level variability, it was not used as the target for the explanatory model.

Instead, the model uses raw percentage error (value added) as the target:

  * pct_error = (actual - predicted) / predicted

This decision ensures that the model learns directly from observed over- and underperformance, rather than from a derived metric that incorporates additional adjustments (e.g., sample size penalties and variability scaling).

The composite score remains useful for identifying high-variability groups, but the explanatory model focuses on the underlying performance signal.

In [441]:
targ="inst_wr_mean"
merge_df=driver_df.merge(residual_df[[targ,"unit_id"]], on=["unit_id"],how="right")
merge_df.head()

,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,admission_rate_overall,open_admissions_policy,school_name,unit_id,has_endowment,inst_wr_mean
0,0.0,0.0,0.0,0.0,0.0,0.0,0.1043,0.0313,0.0,0.0,0.1913,0.0,0.0,0.0278,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0765,0.0591,0.033,0.0,0.0,0.4017,0.0748,0.0,19483.0,NaN,NaN,47.0,17.0,NaN,NaN,1.0,NaN,1.0,Atlantic Technical College,132374,0,-2415.625
1,0.0,0.0,0.0,0.0,0.0,0.0,0.1043,0.0313,0.0,0.0,0.1913,0.0,0.0,0.0278,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0765,0.0591,0.033,0.0,0.0,0.4017,0.0748,0.0,19483.0,NaN,NaN,47.0,17.0,NaN,NaN,1.0,NaN,1.0,Atlantic Technical College,132374,0,-2415.625
2,0.0,0.0,0.0,0.0,0.0,0.0,0.1043,0.0313,0.0,0.0,0.1913,0.0,0.0,0.0278,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0765,0.0591,0.033,0.0,0.0,0.4017,0.0748,0.0,19483.0,NaN,NaN,47.0,17.0,NaN,NaN,1.0,NaN,1.0,Atlantic Technical College,132374,0,-2415.625
3,0.0,0.0,0.0,0.0,0.0,0.0,0.1043,0.0313,0.0,0.0,0.1913,0.0,0.0,0.0278,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0765,0.0591,0.033,0.0,0.0,0.4017,0.0748,0.0,19483.0,NaN,NaN,47.0,17.0,NaN,NaN,1.0,NaN,1.0,Atlantic Technical College,132374,0,-2415.625
4,0.0,0.0,0.0,0.0,0.0,0.0,0.1043,0.0313,0.0,0.0,0.1913,0.0,0.0,0.0278,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0765,0.0591,0.033,0.0,0.0,0.4017,0.0748,0.0,19483.0,NaN,NaN,47.0,17.0,NaN,NaN,1.0,NaN,1.0,Atlantic Technical College,132374,0,-2415.625


In [444]:
model_df=merge_df.copy()
print("ipedsd_df shape:", ipedsd_df.shape)
print("residual_df shape:", residual_df.shape)
print("model_df shape:", model_df.shape)

print("model_df columns:")
print(sorted(model_df.columns.tolist()))

ipedsd_df shape: (6909, 27)
residual_df shape: (2052, 29)
model_df shape: (2052, 52)
model_df columns:
['admission_rate_overall', 'dolflag', 'endowment_begin', 'endowment_end', 'faculty_salary', 'ft_faculty_rate', 'has_endowment', 'inst_wr_mean', 'instructional_expenditure_per_fte', 'open_admissions_policy', 'program_percentage_agriculture', 'program_percentage_architecture', 'program_percentage_biological', 'program_percentage_business_marketing', 'program_percentage_communication', 'program_percentage_communications_technology', 'program_percentage_computer', 'program_percentage_construction', 'program_percentage_education', 'program_percentage_engineering', 'program_percentage_engineering_technology', 'program_percentage_english', 'program_percentage_ethnic_cultural_gender', 'program_percentage_family_consumer_science', 'program_percentage_health', 'program_percentage_history', 'program_percentage_humanities', 'program_percentage_language', 'program_percentage_legal', 'program_perce

In [448]:
model_df=model_df.drop(columns=[
    "program_reporter_programs_offered",
    # 'code',
    # 'unit_id',
    # 'school_name'
    ]
)

In [449]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2052 entries, 0 to 2051
Data columns (total 51 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   program_percentage_agriculture                           2051 non-null   float64
 1   program_percentage_resources                             2051 non-null   float64
 2   program_percentage_architecture                          2051 non-null   float64
 3   program_percentage_ethnic_cultural_gender                2051 non-null   float64
 4   program_percentage_communication                         2051 non-null   float64
 5   program_percentage_communications_technology             2051 non-null   float64
 6   program_percentage_computer                              2051 non-null   float64
 7   program_percentage_personal_culinary                     2051 non-null   float64
 8   program_percentage_education

In [450]:
program_cols = [c for c in model_df.columns if c.startswith("program_percentage_")]

for c in program_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0)

# STEM
model_df["pct_stem"] = model_df[
    [
        "program_percentage_computer",
        "program_percentage_engineering",
        "program_percentage_engineering_technology",
        "program_percentage_mathematics",
        "program_percentage_physical_science",
        "program_percentage_biological",
        "program_percentage_science_technology"
    ]
].sum(axis=1)

# Business / Econ
model_df["pct_business"] = model_df[
    ["program_percentage_business_marketing"]
].sum(axis=1)

# Health
model_df["pct_health"] = model_df[
    ["program_percentage_health"]
].sum(axis=1)

# Social Sciences
model_df["pct_social_science"] = model_df[
    [
        "program_percentage_psychology",
        "program_percentage_social_science",
        "program_percentage_history",
        "program_percentage_public_administration_social_service"
    ]
].sum(axis=1)

# Humanities
model_df["pct_humanities"] = model_df[
    [
        "program_percentage_english",
        "program_percentage_language",
        "program_percentage_humanities",
        "program_percentage_philosophy_religious",
        "program_percentage_theology_religious_vocation",
        "program_percentage_ethnic_cultural_gender"
    ]
].sum(axis=1)

# Arts & Communication
model_df["pct_arts_comm"] = model_df[
    [
        "program_percentage_visual_performing",
        "program_percentage_communication"
    ]
].sum(axis=1)

# Education
model_df["pct_education"] = model_df[
    ["program_percentage_education"]
].sum(axis=1)

# Trades / Technical
model_df["pct_trades"] = model_df[
    [
        "program_percentage_construction",
        "program_percentage_mechanic_repair_technology",
        "program_percentage_precision_production",
        "program_percentage_transportation"
    ]
].sum(axis=1)

# Services / Consumer
model_df["pct_services"] = model_df[
    [
        "program_percentage_personal_culinary",
        "program_percentage_family_consumer_science",
        "program_percentage_parks_recreation_fitness"
    ]
].sum(axis=1)

# Law / Security
model_df["pct_law_security"] = model_df[
    [
        "program_percentage_legal",
        "program_percentage_security_law_enforcement"
    ]
].sum(axis=1)

# Agriculture / Natural resources
model_df["pct_agriculture"] = model_df[
    [
        "program_percentage_agriculture",
        "program_percentage_resources"
    ]
].sum(axis=1)

model_df["pct_high_roi"] = (
    model_df["program_percentage_engineering"] +
    model_df["program_percentage_computer"] +
    model_df["program_percentage_health"]
)

model_df["pct_low_roi"] = (
    model_df["program_percentage_education"] +
    model_df["program_percentage_personal_culinary"] +
    model_df["program_percentage_humanities"]
)

model_df["program_hhi"] = (model_df[program_cols] ** 2).sum(axis=1)

model_df["max_program_share"] = model_df[program_cols].max(axis=1)

model_df["high_roi_x_concentration"] = (
    model_df["pct_high_roi"] * model_df["program_hhi"]
)

# model_df = model_df.drop(columns=program_cols)



### Feature Selection for Explanatory Model

Variables used in the initial prediction model (e.g., credential level, distance, and other structural constraints) were excluded from the explanatory model.

This is because the residuals already represent performance after controlling for these factors. Including them again would introduce circular reasoning and reduce the interpretability of the results.

Instead, the explanatory model focuses on institutional characteristics and program composition variables that were not used in the prediction stage, allowing us to better understand what drives over- and underperformance.

In [451]:
display("Repeated columns within instituions",(model_df.groupby("unit_id").nunique() > 1).sum())
display("Feature correlation to target variable",model_df.corr()[targ].sort_values())

'Repeated columns within instituions'

program_percentage_agriculture               0
program_percentage_resources                 0
program_percentage_architecture              0
program_percentage_ethnic_cultural_gender    0
program_percentage_communication             0
                                            ..
pct_high_roi                                 0
pct_low_roi                                  0
program_hhi                                  0
max_program_share                            0
high_roi_x_concentration                     0
Length: 66, dtype: int64

C:\Users\sebas\AppData\Local\Temp\ipykernel_27848\703387114.py:2: FutureWarning:

The default value of numeric_only in DataFrame.corr is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.



'Feature correlation to target variable'

program_percentage_theology_religious_vocation   -0.147468
dolflag                                          -0.136250
pct_low_roi                                      -0.108289
admission_rate_overall                           -0.100223
pct_humanities                                   -0.097898
                                                    ...   
endowment_end                                     0.247188
endowment_begin                                   0.247504
program_percentage_visual_performing              0.272596
inst_wr_mean                                      1.000000
program_percentage_library                             NaN
Name: inst_wr_mean, Length: 65, dtype: float64

In [452]:
display("Repeated columns within instituions",(model_df.groupby("unit_id").nunique() > 1).sum())
display("Feature correlation to target variable",model_df.corr()[targ].sort_values())


'Repeated columns within instituions'

program_percentage_agriculture               0
program_percentage_resources                 0
program_percentage_architecture              0
program_percentage_ethnic_cultural_gender    0
program_percentage_communication             0
                                            ..
pct_high_roi                                 0
pct_low_roi                                  0
program_hhi                                  0
max_program_share                            0
high_roi_x_concentration                     0
Length: 66, dtype: int64

C:\Users\sebas\AppData\Local\Temp\ipykernel_27848\47706843.py:2: FutureWarning:

The default value of numeric_only in DataFrame.corr is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.



'Feature correlation to target variable'

program_percentage_theology_religious_vocation   -0.147468
dolflag                                          -0.136250
pct_low_roi                                      -0.108289
admission_rate_overall                           -0.100223
pct_humanities                                   -0.097898
                                                    ...   
endowment_end                                     0.247188
endowment_begin                                   0.247504
program_percentage_visual_performing              0.272596
inst_wr_mean                                      1.000000
program_percentage_library                             NaN
Name: inst_wr_mean, Length: 65, dtype: float64

In [453]:
inst_model_df = model_df.drop_duplicates(subset="unit_id").reset_index(drop=True)
inst_model_df=inst_model_df.drop(columns="unit_id")
print("inst_model_df shape:", inst_model_df.shape)
display(inst_model_df.head())

inst_model_df shape: (237, 66)


,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,student_faculty_ratio,endowment_begin,endowment_end,dolflag,admission_rate_overall,open_admissions_policy,school_name,has_endowment,inst_wr_mean,pct_stem,pct_business,pct_health,pct_social_science,pct_humanities,pct_arts_comm,pct_education,pct_trades,pct_services,pct_law_security,pct_agriculture,pct_high_roi,pct_low_roi,program_hhi,max_program_share,high_roi_x_concentration
0,0.0,0.0000,0.0,0.0,0.0000,0.000,0.1043,0.0313,0.0000,0.0000,0.1913,0.0,0.0,0.0278,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0765,0.0591,0.033,0.0000,0.0000,0.4017,0.0748,0.0000,19483.0,NaN,NaN,17.0,NaN,NaN,1.0,NaN,1.0,Atlantic Technical College,0,-2415.625000,0.2956,0.0748,0.4017,0.0000,0.0000,0.0000,0.0000,0.1686,0.0313,0.0278,0.0000,0.5060,0.0313,0.226619,0.4017,0.114669
1,0.0,0.0000,0.0,0.0,0.0000,0.000,0.0000,0.0000,0.0556,0.0000,0.0000,0.0,0.0,0.0000,0.0370,0.2407,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3519,0.0000,0.0,0.1667,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.1111,0.0000,0.0185,0.0185,9992.0,5515.0,1.0000,8.0,8589732.0,7970708.0,0.0,0.3599,2.0,Baptist University of Florida,1,-7158.445313,0.0000,0.0185,0.0000,0.1852,0.6296,0.1111,0.0556,0.0000,0.0000,0.0000,0.0000,0.0000,0.2963,0.227047,0.3519,0.000000
2,0.0,0.0000,0.0,0.0,0.0366,0.000,0.0539,0.0000,0.0173,0.0000,0.0000,0.0,0.0,0.0077,0.0019,0.0135,0.0,0.1522,0.0019,0.0000,0.0000,0.0405,0.0019,0.0019,0.0096,0.0,0.0636,0.0289,0.0829,0.0385,0.0000,0.0000,0.000,0.0000,0.0366,0.2216,0.1888,0.0000,11546.0,8861.0,0.2830,7.0,53988781.0,58752320.0,0.0,0.7724,2.0,Barry University,1,4972.134001,0.2176,0.1888,0.2216,0.1850,0.0192,0.0732,0.0173,0.0000,0.0405,0.0366,0.0000,0.2755,0.0308,0.129024,0.2216,0.035546
3,0.0,0.0034,0.0,0.0,0.1000,0.000,0.0207,0.0000,0.0310,0.0138,0.0000,0.0,0.0,0.0000,0.0069,0.2207,0.0,0.0448,0.0034,0.0000,0.0138,0.1000,0.0000,0.0000,0.0000,0.0,0.1414,0.0793,0.0000,0.0310,0.0000,0.0000,0.000,0.0000,0.0172,0.0138,0.1586,0.0000,11053.0,7167.0,0.7600,17.0,36303752.0,41110832.0,1.0,0.8824,2.0,Bethune-Cookman University,1,-8228.132254,0.0827,0.1586,0.0138,0.1724,0.2276,0.1172,0.0310,0.0000,0.1000,0.0793,0.0034,0.0483,0.2517,0.125440,0.2207,0.006059
4,0.0,0.0051,0.0,0.0,0.0696,0.017,0.0000,0.0000,0.0357,0.0000,0.0000,0.0,0.0,0.0000,0.0000,0.0017,0.0,0.0628,0.0000,0.0136,0.0068,0.0696,0.0000,0.0000,0.0000,0.0,0.0951,0.0594,0.0000,0.0119,0.0000,0.0000,0.000,0.0611,0.1070,0.0051,0.3786,0.0000,7523.0,6894.0,0.4331,16.0,37840050.0,42918486.0,0.0,0.7347,2.0,Lynn University,1,-1361.238281,0.0628,0.3786,0.0051,0.1070,0.0017,0.1766,0.0357,0.061

In [454]:
display(inst_model_df.describe())
display("Feature correlation to target variable",inst_model_df.corr()[targ].sort_values())
display(inst_model_df.info())

,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,student_faculty_ratio,endowment_begin,endowment_end,dolflag,admission_rate_overall,open_admissions_policy,has_endowment,inst_wr_mean,pct_stem,pct_business,pct_health,pct_social_science,pct_humanities,pct_arts_comm,pct_education,pct_trades,pct_services,pct_law_security,pct_agriculture,pct_high_roi,pct_low_roi,program_hhi,max_program_share,high_roi_x_concentration
count,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.0,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,131.000000,119.000000,234.000000,6.900000e+01,6.900000e+01,232.000000,60.000000,236.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000,237.000000
mean,0.005046,0.002768,0.000428,0.000129,0.012111,0.005585,0.026908,0.247657,0.008358,0.009023,0.018696,0.000576,0.007512,0.001341,0.002105,0.061309,0.0,0.011784,0.000843,0.000161,0.007109,0.014222,0.000721,0.005443,0.001458,0.001217,0.017682,0.028503,0.002323,0.007206,0.013909,0.044054,0.014479,0.015509,0.018222,0.313573,0.062561,0.001027,7094.898734,7523.137405,0.547262,17.290598,1.439028e+08,1.574963e+08,0.435345,0.691170,1.271186,0.291139,711.908489,0.069930,0.062561,0.313573,0.028238,0.070282,0.030333,0.008358,0.087951,0.269391,0.029843,0.007814,0.349505,0.317324,0.583011,0.663747,0.240639
std,0.026817,0.012081,0.002930,0.000690,0.063302,0.036492,0.053320,0.391466,0.020508,0.044962,0.061817,0.004072,0.049691,0.004185,0.006956,0.147734,0.0,0.043286,0.002920,0.001664,0.035960,0.094370,0.004855,0.041925,0.006023,0.008082,0.068044,0.067601,0.009408,0.023873,0.071381,0.152304,0.071836,0.068561,0.087292,0.366474,0.120542,0.003753,4890.229484,2254.080825,0.288060,8.197728,3.449602e+08,3.729640e+08,0.496874,0.224612,0.445517,0.455249,6182.632521,0.116585,0.120542,0.366474,0.083010,0.156853,0.109770,0.020508,0.210639,0.393731,0.067893,0.029104,0.359565,0.379384,0.355403,0.308594,0.365409
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0

C:\Users\sebas\AppData\Local\Temp\ipykernel_27848\3989557815.py:2: FutureWarning:

The default value of numeric_only in DataFrame.corr is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.



'Feature correlation to target variable'

program_percentage_theology_religious_vocation   -0.166047
open_admissions_policy                           -0.107641
pct_humanities                                   -0.101545
program_percentage_multidiscipline               -0.088778
program_percentage_philosophy_religious          -0.083364
                                                    ...   
faculty_salary                                    0.220181
pct_arts_comm                                     0.321536
program_percentage_visual_performing              0.423851
inst_wr_mean                                      1.000000
program_percentage_library                             NaN
Name: inst_wr_mean, Length: 65, dtype: float64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 66 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   program_percentage_agriculture                           237 non-null    float64
 1   program_percentage_resources                             237 non-null    float64
 2   program_percentage_architecture                          237 non-null    float64
 3   program_percentage_ethnic_cultural_gender                237 non-null    float64
 4   program_percentage_communication                         237 non-null    float64
 5   program_percentage_communications_technology             237 non-null    float64
 6   program_percentage_computer                              237 non-null    float64
 7   program_percentage_personal_culinary                     237 non-null    float64
 8   program_percentage_education  

None

In [455]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

# Reproducibility
RANDOM_STATE = 42

In [474]:
# -------------------
# 1. Define target
# -------------------
raw_target_col = targ

# Drop target and target-like columns from X
drop_cols = [
    raw_target_col,
    "inst_wr_mean",   # important: likely leakage / near-leakage
    "unit_id"         # ID, not a feature
]

X = inst_model_df.drop(columns=drop_cols, errors="ignore").copy()
y = pd.to_numeric(inst_model_df[raw_target_col], errors="coerce").copy()

# Force predictors numeric
X = X.apply(pd.to_numeric, errors="coerce")

# -------------------
# 2. Train/test split
# -------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# -------------------
# 3. Clip target using TRAINING quantiles only
# -------------------
lower = y_train.quantile(0.01)
upper = y_train.quantile(0.99)

y_train_clip = y_train.clip(lower, upper)
y_test_clip = y_test.clip(lower, upper)

# -------------------
# 4. Recompute numeric columns
# -------------------
num_cols = X_train.columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols)
])

# -------------------
# 5. Ridge model
# -------------------
ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Ridge())
])

ridge_model = TransformedTargetRegressor(
    regressor=ridge_pipe,
    transformer=PowerTransformer(method="yeo-johnson", standardize=False)
)

ridge_param_grid = {
    "regressor__reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0, 100.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ridge_grid = GridSearchCV(
    estimator=ridge_model,
    param_grid=ridge_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

ridge_grid.fit(X_train, y_train_clip)

ridge_best = ridge_grid.best_estimator_
ridge_preds = ridge_best.predict(X_test)

print("RIDGE")
print("Best params:", ridge_grid.best_params_)
print("Best CV MAE:", round(-ridge_grid.best_score_, 4))
print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, ridge_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, ridge_preds), 4))

print("Test MAE (vs clipped y_test):", round(mean_absolute_error(y_test_clip, ridge_preds), 4))
print("Test R2 (vs clipped y_test):", round(r2_score(y_test_clip, ridge_preds), 4))

# feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
# print(feature_names)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
RIDGE
Best params: {'regressor__reg__alpha': 100.0}
Best CV MAE: 3305.3954
Test MAE (vs unclipped y_test): 3910.9879
Test R2 (vs unclipped y_test): -0.0011
Test MAE (vs clipped y_test): 3170.886
Test R2 (vs clipped y_test): -0.1537


In [475]:
enet_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", ElasticNet(max_iter=100000))
])

enet_model = TransformedTargetRegressor(
    regressor=enet_pipe,
    transformer=PowerTransformer(method="yeo-johnson", standardize=False)
)

enet_param_grid = {
    "regressor__reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0],
    "regressor__reg__l1_ratio": [0.005,0.1, 0.3, 0.5, 0.7, 0.9]
}

enet_grid = GridSearchCV(
    estimator=enet_model,
    param_grid=enet_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

enet_grid.fit(X_train, y_train_clip)

enet_best = enet_grid.best_estimator_
enet_preds = enet_best.predict(X_test)

print("\nELASTIC NET")
print("Best params:", enet_grid.best_params_)
print("Best CV MAE:", round(-enet_grid.best_score_, 4))
print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, enet_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, enet_preds), 4))
print("Test MAE (vs clipped y_test):", round(mean_absolute_error(y_test_clip, enet_preds), 4))
print("Test R2 (vs clipped y_test):", round(r2_score(y_test_clip, enet_preds), 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits

ELASTIC NET
Best params: {'regressor__reg__alpha': 10.0, 'regressor__reg__l1_ratio': 0.9}
Best CV MAE: 3289.5494
Test MAE (vs unclipped y_test): 3712.1038
Test R2 (vs unclipped y_test): 0.0167
Test MAE (vs clipped y_test): 2972.0019
Test R2 (vs clipped y_test): -0.0663


In [478]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor( 
    max_depth=3,
    learning_rate=0.05,
    max_iter=200,
    random_state=42 
    ) 
model.fit(X_train, y_train) 
preds = model.predict(X_test) 
print("MAE:", mean_absolute_error(y_test, preds)) 
print("R2:", r2_score(y_test, preds))

MAE: 3888.8502116059135
R2: 0.07796312873003963


In [493]:
model = HistGradientBoostingRegressor(
    max_depth=5,              # allow more interactions
    learning_rate=0.01,       # slower learning
    max_iter=750,             # more trees
    # min_samples_leaf=10,      # regularization
    l2_regularization=2.0,    # stabilize
    random_state=42
)

model.fit(X_train, y_train) 
preds = model.predict(X_test) 
print("MAE:", mean_absolute_error(y_test, preds)) 
print("R2:", r2_score(y_test, preds))

MAE: 4096.317770116216
R2: 0.07313627269929113


In [499]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

importances = pd.Series(
    result.importances_mean,
    index=X.columns
).sort_values(ascending=False)

print(importances.head(10))

ft_faculty_rate                              7.788880e+06
admission_rate_overall                       7.601033e+06
pct_services                                 6.852232e+06
program_percentage_humanities                3.769050e+05
program_percentage_computer                  2.751435e+05
pct_agriculture                              2.126284e+05
program_percentage_engineering_technology    1.542135e+05
dolflag                                      1.219050e+05
pct_stem                                     1.189400e+05
pct_arts_comm                                1.084858e+05
dtype: float64


In [494]:
from sklearn.linear_model import LogisticRegression
# -------------------
# 1. Target (top vs bottom)
# -------------------
y_raw = inst_model_df["inst_wr_mean"].copy()

low = y_raw.quantile(0.25)
high = y_raw.quantile(0.75)

mask = (y_raw <= low) | (y_raw >= high)

# -------------------
# 2. Use ALL columns except target
# -------------------
X_clf = inst_model_df.drop(columns=["inst_wr_mean"], errors="ignore").loc[mask].copy()
y_clf = (y_raw.loc[mask] >= high).astype(int)

# force everything numeric
X_clf = X_clf.apply(pd.to_numeric, errors="coerce")

# -------------------
# 3. Train/test split
# -------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

# -------------------
# 4. Preprocessing (CRITICAL)
# -------------------
num_cols = X_train.columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols)
])

# -------------------
# 5. Model
# -------------------
clf = Pipeline([
    ("preprocessor", preprocessor),
    ("logit", LogisticRegression(max_iter=10000))
])

# -------------------
# 6. Fit
# -------------------
clf.fit(X_train, y_train)

# -------------------
# 7. Evaluate
# -------------------
preds = clf.predict(X_test)
probs = clf.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("ROC AUC:", round(roc_auc_score(y_test, probs), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))

Accuracy: 0.4167
ROC AUC: 0.3958

Confusion Matrix:
 [[6 6]
 [8 4]]

Classification Report:
               precision    recall  f1-score   support

           0       0.43      0.50      0.46        12
           1       0.40      0.33      0.36        12

    accuracy                           0.42        24
   macro avg       0.41      0.42      0.41        24
weighted avg       0.41      0.42      0.41        24



In [495]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

clf_hgb = HistGradientBoostingClassifier(
    max_depth=3,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

clf_hgb.fit(X_train, y_train)

preds_hgb = clf_hgb.predict(X_test)
probs_hgb = clf_hgb.predict_proba(X_test)[:, 1]

print("HGB Accuracy:", round(accuracy_score(y_test, preds_hgb), 4))
print("HGB ROC AUC:", round(roc_auc_score(y_test, probs_hgb), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds_hgb))
print("\nClassification Report:\n", classification_report(y_test, preds_hgb))

HGB Accuracy: 0.4583
HGB ROC AUC: 0.4514

Confusion Matrix:
 [[7 5]
 [8 4]]

Classification Report:
               precision    recall  f1-score   support

           0       0.47      0.58      0.52        12
           1       0.44      0.33      0.38        12

    accuracy                           0.46        24
   macro avg       0.46      0.46      0.45        24
weighted avg       0.46      0.46      0.45        24



In [496]:
from sklearn.model_selection import cross_val_score

auc_scores = cross_val_score(
    clf,
    X_clf,
    y_clf,
    cv=5,
    scoring="roc_auc"
)

print("AUC mean:", auc_scores.mean())
print("AUC std:", auc_scores.std())

AUC mean: 0.5138888888888888
AUC std: 0.04457446259967005


In [497]:
from sklearn.model_selection import cross_val_score

auc_scores = cross_val_score(
    clf_hgb,
    X_clf,
    y_clf,
    cv=5,
    scoring="roc_auc"
)

print("AUC mean:", auc_scores.mean())
print("AUC std:", auc_scores.std())

AUC mean: 0.45
AUC std: 0.1516625966079855
